Modelo de planeación multietapa \
Juan Fernando Cárdenas Duque – 20251020038 \
Nicolás Olivares Chica – 20251020008 \
María José Polanco Charry - 20251020046


In [7]:
!apt-get update && apt-get install -y coinor-cbc

"apt-get" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [8]:
!pip install "pulp[cbc]"

"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [9]:
import pulp
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import networkx as nx
# La resolución del Paper y la de este modelo difieren en 2 unidades monetarias
# debido al modelo de solución; sin embargo, el modelo está montado tal como se muestra en el Paper.
# Además, algunos datos difieren porque el modelo tiene múltiples soluciones en
# algunas variables. Sin embargo, el costo mínimo sigue siendo el mismo.

# =====================================================================
# 1. DATOS
# =====================================================================
# Convención: i = etapa, j = fuente (máquina) dentro de la etapa, t = periodo (1..T)

T_periodos = 4   # número de periodos (trimestres)
M_etapas = 4     # número de etapas del proceso

# Demanda de producto terminado por periodo
demanda = {1: 50500, 2: 65500, 3: 82000, 4: 92000}

# Inventario de producto terminado
I_plus_inicial = 1000   # K1
I_plus_final = 1000     # K2
I_minus_inicial = 0     # faltantes al inicio
I_minus_final = 0       # faltantes al final

# Costo por unidad pendiente / escasez
psi_escasez = 4.0

# Costo unitario de producción C[i,j] = [T1, T2, T3, T4]
costo_prod_base = {
    (1, 1): [25, 28, 30, 28], (1, 2): [28, 30, 30, 32], (1, 3): [30, 32, 34, 32],
    (2, 1): [45, 46, 44, 48], (2, 2): [48, 50, 49, 50],
    (3, 1): [12, 15, 18, 19], (3, 2): [14, 16, 18, 20], (3, 3): [13, 16, 20, 22],
    (4, 1): [24, 29, 24, 28], (4, 2): [24, 29, 24, 27],
    (4, 3): [26, 30, 27, 25], (4, 4): [28, 33, 27, 25],
}

# Costo de preparación (setup) A[i,j], igual en todos los periodos
costo_setup_base = {
    (1, 1): 25000, (1, 2): 24000, (1, 3): 20000,
    (2, 1): 24000, (2, 2): 22000,
    (3, 1): 12000, (3, 2): 10000, (3, 3): 12000,
    (4, 1): 23000, (4, 2): 25000, (4, 3): 23000, (4, 4): 22000,
}

# Capacidad en unidades de producto Cd[i,j] = [T1, T2, T3, T4]
# Datos dados como capacidad en unidades por hora/trimestre.
# Capacidad disponible hora/trimestre (tiempo de empleo) * Tasa de producción de unidades por hora
capacidad_base = {
    (1, 1): [39952, 40592, 45712, 39952],
    (1, 2): [50510, 51310, 57710, 50510],
    (1, 3): [112537.6, 114329.6, 128665.6, 112537.6],
    (2, 1): [165782.4, 168470.4, 189974.4, 165782.4],
    (2, 2): [171670.4, 174486.4, 197014.4, 171670.4],
    (3, 1): [39952, 40592, 45712, 39952],
    (3, 2): [55832, 56728, 63896, 55832],
    (3, 3): [130812, 132924, 149820, 130812],
    (4, 1): [28713.6, 29174.4, 32860.8, 28713.6],
    (4, 2): [40192, 40832, 45952, 40192],
    (4, 3): [56571.2, 57467.2, 64635.2, 56571.2],
    (4, 4): [72259.2, 73411.2, 82627.2, 72259.2],
}

# Costo de mantener inventario
h_proceso = {1: 2.0, 2: 3.0, 3: 4.0}   # inventario en proceso, etapas 1..M-1
h_final = 5.0                          # producto terminado (etapa M)

# Solver (msg=True muestra el registro interno del solver)
SOLVER = pulp.HiGHS(msg=False)


# =====================================================================
# 2. PREPARACIÓN DE DATOS
# =====================================================================
# Fuentes por etapa, derivadas de la tabla de costos
J_fuentes = {i: sorted(j for (ii, j) in costo_prod_base if ii == i)
             for i in range(1, M_etapas + 1)}

# Verificación: las tres tablas deben tener exactamente las mismas fuentes
assert set(costo_prod_base) == set(costo_setup_base) == set(capacidad_base), \
    "Las tablas de costo, setup y capacidad deben tener las mismas fuentes (i, j)"
for tabla in (costo_prod_base, capacidad_base):
    for clave, lista in tabla.items():
        assert len(lista) == T_periodos, f"{clave} debe tener {T_periodos} valores"

# Expansión a la forma [i, j, t] que usa el modelo
costo_produccion = {(i, j, t): costo_prod_base[i, j][t - 1]
                    for (i, j) in costo_prod_base for t in range(1, T_periodos + 1)}
costo_setup = {(i, j, t): costo_setup_base[i, j]
               for (i, j) in costo_setup_base for t in range(1, T_periodos + 1)}
capacidad = {(i, j, t): capacidad_base[i, j][t - 1]
             for (i, j) in capacidad_base for t in range(1, T_periodos + 1)}
costo_inventario_proceso = {(i, t): h_proceso[i]
                            for i in range(1, M_etapas) for t in range(1, T_periodos + 1)}
costo_inventario_final = {t: h_final for t in range(1, T_periodos + 1)}

ETAPAS = range(1, M_etapas + 1)
PERIODOS = range(1, T_periodos + 1)
FUENTES = [(i, j) for i in ETAPAS for j in J_fuentes[i]]


# =====================================================================
# 3. MODELO
# =====================================================================
modelo = pulp.LpProblem("Planeacion_Agregada_Multietapa_Multifuente", pulp.LpMinimize)

# --- Variables de decisión ---
# X[i,j,t]: unidades producidas (entera >= 0)
X = {(i, j, t): modelo.add_variable(f"X_{i}_{j}_{t}", lowBound=0, cat='Integer')
     for (i, j) in FUENTES for t in PERIODOS}

# Y[i,j,t]: 1 si la fuente se alista (setup) en ese periodo
Y = {(i, j, t): modelo.add_variable(f"Y_{i}_{j}_{t}", cat='Binary')
     for (i, j) in FUENTES for t in PERIODOS}

# I_proceso[i,t]: inventario en proceso, etapas 1..M-1, t = 0..T
I_proceso = {(i, t): modelo.add_variable(f"I_proceso_{i}_{t}", lowBound=0, cat='Integer')
             for i in range(1, M_etapas) for t in range(0, T_periodos + 1)}

# I_plus[t], I_minus[t]: producto terminado y faltantes, t = 0..T
I_plus = {t: modelo.add_variable(f"I_plus_{t}", lowBound=0, cat='Integer')
          for t in range(0, T_periodos + 1)}
I_minus = {t: modelo.add_variable(f"I_minus_{t}", lowBound=0, cat='Integer')
           for t in range(0, T_periodos + 1)}

# --- Función objetivo: producción + setup + inventarios + escasez ---
modelo += (
    pulp.lpSum(costo_produccion[i, j, t] * X[i, j, t] + costo_setup[i, j, t] * Y[i, j, t]
               for (i, j) in FUENTES for t in PERIODOS)
    + pulp.lpSum(costo_inventario_proceso[i, t] * I_proceso[i, t]
                 for i in range(1, M_etapas) for t in PERIODOS)
    + pulp.lpSum(costo_inventario_final[t] * I_plus[t] for t in PERIODOS)
    + pulp.lpSum(psi_escasez * I_minus[t] for t in PERIODOS)
)

# --- Restricciones ---
# A. Condiciones iniciales y finales
modelo += I_plus[0] == I_plus_inicial # El inventario terminado entra al inicio del periodo t
modelo += I_plus[T_periodos] == I_plus_final # Al final debe haber I_plus_final unidades al cabo de los T periodos
modelo += I_minus[0] == I_minus_inicial # Mismo caso, solo que con ´rdenes pendientes
modelo += I_minus[T_periodos] == I_minus_final
for i in range(1, M_etapas):
    modelo += I_proceso[i, 0] == 0 # No deben haber unidades en proceso al inicio
    modelo += I_proceso[i, T_periodos] == 0 # Ni al final

# B. Capacidad ligada al setup: X <= Cd * Y
for (i, j) in FUENTES:
    for t in PERIODOS:
        modelo += X[i, j, t] <= capacidad[i, j, t] * Y[i, j, t] # Cantidades no pueden ser mayores a la capacidad máxima con alistamiento

# C. Balance de inventario en etapas intermedias (i = 1..M-1)
for i in range(1, M_etapas):
    for t in PERIODOS:
        produccion = pulp.lpSum(X[i, j, t] for j in J_fuentes[i])
        consumo = pulp.lpSum(X[i + 1, j, t] for j in J_fuentes[i + 1])
        modelo += produccion + I_proceso[i, t - 1] - consumo - I_proceso[i, t] == 0

# D. Balance de la etapa final y cumplimiento de la demanda
for t in PERIODOS:
    produccion_final = pulp.lpSum(X[M_etapas, j, t] for j in J_fuentes[M_etapas])
    modelo += (produccion_final + I_plus[t - 1] - I_plus[t]
               + I_minus[t] - I_minus[t - 1] == demanda[t])


# =====================================================================
# 4. RESOLUCIÓN Y RESULTADOS
# =====================================================================
print("Optimizando modelo...")
stats = modelo.solve(SOLVER)

print(f"\nEstado: {stats.status_str}")
print(f"Costo total mínimo: ${pulp.value(modelo.objective):,.2f}\n")

item_idx = 1

print("Etapa i, fuente j, periodo t")
# 1. Variables de Producción (X_i_j_t) ordenadas por Etapa, Periodo y Fuente
print("\n[VARIABLES DE PRODUCCIÓN - X_i_j_t]")
for i in ETAPAS:
    for t in PERIODOS:
        for j in J_fuentes[i]:
            var = X[i, j, t]
            print(f"{item_idx:3d}. {var.name} = {var.varValue:,.2f}")
            item_idx += 1

# 2. Inventario en Proceso (I_proceso)
print("\n[INVENTARIO EN PROCESO - I_proceso_i_t]")
for i in range(1, M_etapas):
    for t in PERIODOS:
        var = I_proceso[i, t]
        print(f"{item_idx:3d}. {var.name} = {var.varValue:,.2f}")
        item_idx += 1

# 3. Inventario Final de Producto Terminado (I_plus) y Escasez (I_minus)
print("\n[INVENTARIO FINAL Y ESCASEZ]")
for t in PERIODOS:
    var = I_plus[t]
    print(f"{item_idx:3d}. {var.name} = {var.varValue:,.2f}")
    item_idx += 1
for t in PERIODOS:
    var = I_minus[t]
    print(f"{item_idx:3d}. {var.name} = {var.varValue:,.2f}")
    item_idx += 1

# 4. Variables Binarias de Setup (Y_i_j_t)
print("\n[VARIABLES DE SETUP - Y]")
for i in ETAPAS:
    for t in PERIODOS:
        for j in J_fuentes[i]:
            var = Y[i, j, t]
            val = int(round(var.varValue)) if var.varValue is not None else 0
            print(f"{item_idx:3d}. {var.name} = {val}")
            item_idx += 1
G_sol = nx.DiGraph()

# 1. Añadir nodos de estados por etapa y periodo
for i in ETAPAS:
  for t in PERIODOS:
    G_sol.add_node(f"E{i}\nT{t}", pos=(t, -i))

# Listas para clasificar arcos activos y sus etiquetas
arcos_produccion = []
arcos_inventario = []
etiquetas_arcos = {}

# 2. Extraer flujos activos de producción entre etapas (i -> i+1)
for t in PERIODOS:
  for i in range(1, M_etapas):
    detalles = []
    for j in J_fuentes[i]:
      val = X[i, j, t].varValue
      if val is not None and val > 0.5:
        detalles.append(f"F{j}:{int(round(val)):,}")
    if detalles:
      u, v = f"E{i}\nT{t}", f"E{i+1}\nT{t}"
      G_sol.add_edge(u, v)
      arcos_produccion.append((u, v))
      etiquetas_arcos[(u, v)] = "\n".join(detalles)

# 3. Extraer flujos activos de inventario entre periodos (t -> t+1)
for t in range(1, T_periodos):
  # Inventario en proceso (etapas 1 a M-1)
  for i in range(1, M_etapas):
    inv = I_proceso[i, t].varValue
    if inv is not None and inv > 0.5:
      u, v = f"E{i}\nT{t}", f"E{i}\nT{t+1}"
      G_sol.add_edge(u, v)
      arcos_inventario.append((u, v))
      etiquetas_arcos[(u, v)] = f"Inv:{int(round(inv)):,}"

  # Inventario de producto terminado (etapa M)
  inv_f = I_plus[t].varValue
  if inv_f is not None and inv_f > 0.5:
    u, v = f"E{M_etapas}\nT{t}", f"E{M_etapas}\nT{t+1}"
    G_sol.add_edge(u, v)
    arcos_inventario.append((u, v))
    etiquetas_arcos[(u, v)] = f"Inv:{int(round(inv_f)):,}"

# 4. Configurar visualización
pos = nx.get_node_attributes(G_sol, "pos")
plt.figure(figsize=(14, 9))

# Dibujar nodos
nx.draw_networkx_nodes(
    G_sol,
    pos,
    node_size=2200,
    node_color="#e1f5fe",
    edgecolors="#0277bd",
    linewidths=2,
)
nx.draw_networkx_labels(
    G_sol, pos, font_size=9, font_weight="bold", font_color="#01579b"
)

# Dibujar arcos de producción (azul) e inventario (naranja) solo donde hay flujo > 0
nx.draw_networkx_edges(
    G_sol,
    pos,
    edgelist=arcos_produccion,
    edge_color="#0288d1",
    width=2.2,
    arrowsize=20,
)
nx.draw_networkx_edges(
    G_sol,
    pos,
    edgelist=arcos_inventario,
    edge_color="#f57c00",
    width=2.2,
    style="dashed",
    arrowsize=20,
)

# Añadir etiquetas con cantidades exactas sobre los arcos activos
nx.draw_networkx_edge_labels(
    G_sol,
    pos,
    edge_labels=etiquetas_arcos,
    font_size=8,
    font_weight="bold",
    bbox=dict(
        boxstyle="round,pad=0.2", fc="white", ec="lightgray", alpha=0.9
    ),
)

# Añadir anotaciones de producción de la etapa final (Etapa 4) y demanda por periodo
for t in PERIODOS:
  prod_e4 = [
      f"F{j}:{int(round(X[M_etapas, j, t].varValue)):,}"
      for j in J_fuentes[M_etapas]
      if X[M_etapas, j, t].varValue > 0.5
  ]
  texto_e4 = "Prod E4: " + (", ".join(prod_e4) if prod_e4 else "0")
  plt.text(
      t,
      -M_etapas - 0.35,
      f"{texto_e4}\nDemanda: {demanda[t]:,}",
      ha="center",
      va="top",
      fontsize=8.5,
      bbox=dict(boxstyle="round,pad=0.3", fc="#e8f5e9", ec="#2e7d32"),
  )

# Leyenda
leyenda_prod = mlines.Line2D(
    [], [], color="#0288d1", linewidth=2.2, label="Producción activa (X > 0)"
)
leyenda_inv = mlines.Line2D(
    [],
    [],
    color="#f57c00",
    linewidth=2.2,
    linestyle="--",
    label="Inventario guardado (I > 0)",
)
plt.legend(handles=[leyenda_prod, leyenda_inv], loc="upper right", fontsize=10)

plt.title(
    "Grafo de la Solución Óptima: Flujos Activos de Producción e Inventario",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
plt.xlim(0.4, T_periodos + 0.6)
plt.ylim(-M_etapas - 0.85, -0.4)
plt.axis("off")
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'pulp'